In [0]:
-- ============================================
-- GOLD COMPANY DIMENSION
-- Source: silver.sec_companies
-- Target: gold.dim_company
-- ============================================


-- 1. Create the Gold dimension table
CREATE TABLE IF NOT EXISTS gold.dim_company (
    company_key BIGINT GENERATED ALWAYS AS IDENTITY,

    cik STRING NOT NULL,
    ticker STRING NOT NULL,
    company_name STRING NOT NULL,
    sic_code STRING,
    sic_description STRING,
    state_of_incorporation STRING,
    fiscal_year_end STRING,
    sec_entity_type STRING,

    is_active BOOLEAN DEFAULT TRUE,

    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),

    CONSTRAINT pk_dim_company PRIMARY KEY (company_key)
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');


-- 2. Remove duplicate rows, keeping one row per CIK
DELETE FROM gold.dim_company
WHERE company_key NOT IN (
    SELECT MIN(company_key)
    FROM gold.dim_company
    GROUP BY cik
);


-- 3. Merge Silver company data into Gold
MERGE INTO gold.dim_company AS target

USING (
    SELECT
        cik,
        ticker,
        company_name,
        sic_code,
        sic_description,
        state_of_incorporation,
        fiscal_year_end,
        sec_entity_type
    FROM (
        SELECT
            cik,
            ticker,
            company_name,
            sic_code,
            sic_description,
            state_of_incorporation,
            fiscal_year_end,
            sec_entity_type,
            ROW_NUMBER() OVER (PARTITION BY cik ORDER BY ticker) AS rn
        FROM silver.sec_companies
    )
    WHERE rn = 1
) AS source

ON target.cik = source.cik

WHEN MATCHED THEN
    UPDATE SET
        target.ticker = source.ticker,
        target.company_name = source.company_name,
        target.sic_code = source.sic_code,
        target.sic_description = source.sic_description,
        target.state_of_incorporation = source.state_of_incorporation,
        target.fiscal_year_end = source.fiscal_year_end,
        target.sec_entity_type = source.sec_entity_type,
        target.is_active = TRUE,
        target.updated_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN
    INSERT (
        cik,
        ticker,
        company_name,
        sic_code,
        sic_description,
        state_of_incorporation,
        fiscal_year_end,
        sec_entity_type,
        is_active
    )
    VALUES (
        source.cik,
        source.ticker,
        source.company_name,
        source.sic_code,
        source.sic_description,
        source.state_of_incorporation,
        source.fiscal_year_end,
        source.sec_entity_type,
        TRUE
    );


-- 4. Validate the Gold dimension
SELECT
    company_key,
    cik,
    ticker,
    company_name,
    sic_code,
    sic_description,
    state_of_incorporation,
    fiscal_year_end,
    sec_entity_type,
    is_active,
    created_at,
    updated_at
FROM gold.dim_company
ORDER BY company_key;